In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import numpy as np
from parse_mat3di import parse_mat3di
import evoten
from scipy.linalg import logm

evoten.set_backend("tensorflow")

2026-05-26 14:15:35.191785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779797735.208405  763817 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779797735.213528  763817 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779797735.227044  763817 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779797735.227057  763817 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779797735.227059  763817 computation_placer.cc:177] computation placer alr

In [2]:
odds_ratios, pi, alphabet = parse_mat3di("mat3di.out")

In [3]:
P = odds_ratios * pi # P(x | y)

In [4]:
P.sum(-1)

array([0.999998  , 0.99999917, 0.9999977 , 0.99999535, 0.9999985 ,
       0.9999978 , 0.99999845, 0.99999654, 0.9999969 , 0.999999  ,
       0.9999969 , 0.99999887, 0.9999992 , 0.9999996 , 0.9999988 ,
       0.9999984 , 0.99999744, 0.9999998 , 0.9999974 , 0.99999714],
      dtype=float32)

In [5]:
# re-normalize P for a better tQ estimate
P = P / P.sum(-1, keepdims=True)

In [6]:
tQ = logm(P)

/home/felix/miniforge3/envs/learnMSAdev2/lib/python3.12/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 1.15230284336576e-06
  return f(*arrays, *other_args, **kwargs)


In [7]:
tQ.sum(axis=1) # should be ~0

array([ 1.45147497e-07, -3.93766548e-07,  1.05406803e-06,  2.02539610e-06,
       -1.03429103e-06, -6.30068554e-07,  2.05175119e-07, -2.87026145e-08,
        3.71017899e-07, -1.99886850e-07,  6.95041294e-07, -8.80858932e-07,
        1.44979155e-07, -3.54500450e-08,  1.01248266e-06, -4.15309833e-07,
        8.89721067e-07, -6.94267710e-07,  2.40156235e-07,  4.61092363e-08])

In [8]:
# Some off-diagonals are negative, but close to zero
# Might be due to numerical issues in logm
tQ

array([[-1.00254454e+00,  1.17556165e-02,  3.11868779e-01,
         1.15327553e-01,  2.00772909e-01,  2.40528059e-02,
         1.82384375e-02,  1.44149565e-04,  6.92422544e-03,
         1.66760782e-02, -7.15729172e-05,  2.34237055e-04,
         8.27548955e-02,  1.65065823e-01,  1.44020593e-03,
         2.55723107e-03,  4.92618781e-04, -1.15123124e-02,
         3.50911603e-02,  2.07318395e-02],
       [ 1.87394100e-02, -1.46640876e+00,  7.34973337e-02,
         2.23945391e-04, -1.25156991e-02,  1.18333712e-02,
         6.05093837e-03, -1.49471595e-03, -1.03518862e-03,
         1.70389671e-01,  1.07131758e-04,  4.19298003e-02,
         1.60188142e-01,  2.02198630e-01,  3.04248388e-02,
         1.10953519e-01, -1.45759214e-03,  6.53985714e-01,
         3.08280602e-03, -6.93684893e-04],
       [ 1.51035417e-01,  2.23288291e-02, -1.02783199e+00,
         1.05193934e-02,  4.99510763e-02,  1.34345689e-01,
         1.34072626e-01,  1.90304724e-02,  3.12849059e-03,
         8.69747912e-03,  6.4

In [9]:
tQ_fixed = tQ.copy()

# Zero out negative off-diagonal entries
mask = np.ones(tQ.shape, dtype=bool)
np.fill_diagonal(mask, False)
tQ_fixed[mask & (tQ_fixed < 0)] = 0

# Restore row sums to exactly 0
np.fill_diagonal(tQ_fixed, 0)
np.fill_diagonal(tQ_fixed, -tQ_fixed.sum(axis=1))

In [10]:
def exchangeabilities_from_rates(Qt: np.ndarray, pi: np.ndarray) -> np.ndarray:
    """Recover exchangeabilities from a rate matrix and equilibrium.

    Inverse of make_rate_matrix(..., normalized=False):
        R[i,j] = Qt[i,j] / pi[j]  for i != j
    """
    R = Qt / pi          # broadcast: divides each column j by pi[j]
    # zeroing diagonal
    np.fill_diagonal(R, 0.0)
    return R

In [11]:
R = exchangeabilities_from_rates(tQ_fixed, pi)

In [12]:
print("Symmetry residual:", np.max(np.abs(R - R.T)))

Symmetry residual: 2.185596125769962e-05


In [13]:
Q = evoten.backend.make_rate_matrix(R, pi, normalized=False)

2026-05-26 14:15:37.228768: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-05-26 14:15:37.228794: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2026-05-26 14:15:37.228801: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-05-26 14:15:37.228804: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-05-26 14:15:37.228808: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: powergreif
2026-05-26 14:15:37.228811: I external/local_xla/xla/stream_executor/cuda/cu

In [14]:
print("Round-trip residual:", np.max(np.abs(Q - tQ_fixed)))

Round-trip residual: 1.1920929e-07


In [ ]:
# Looks good!
# Safe to file
evoten.util.write_rate_model("../../evoten/data/foldseek_3Di.model", R, pi)